# multi_15m_strat_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (10).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `15m` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, adaptive_intraday_momentum_signal, process_ticker, backtest_trades, calculate_metrics, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/result.csv')
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])
'''
# Define the date range
start_date = '2016-01-01'
end_date = '2023-12-30'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BEL', 'BHARTIARTL',
       'BPCL', 'BRITANNIA', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT',
       'GRASIM', 'HCLTECH', 'HEROMOTOCO', 'HINDALCO', 'HINDUNILVR',
       'ICICIBANK', 'INDUSINDBK', 'INFY', 'ITC', 'JSWSTEEL', 'KOTAKBANK',
       'LT', 'MARUTI', 'MM', 'NESTLEIND', 'NTPC', 'ONGC', 'POWERGRID',
       'RELIANCE', 'SBILIFE', 'SBIN', 'SHRIRAMFIN', 'SUNPHARMA',
       'TATACONSUM', 'TATAMOTORS', 'TATASTEEL', 'TCS', 'TECHM', 'TITAN',
       'TRENT', 'ULTRACEMCO', 'WIPRO']  # Add tickers you wa

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data'''

In [ ]:
import pandas as pd
'''
# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('15min', offset='15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data'''

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# === Strategy Function ===
def adaptive_intraday_momentum_signal(df, base_window=15, min_window=5,
                                                 vol_spike_factor=1.5, vol_increase_thresh=0.3,
                                                 momentum_thresh=0.001, vol_ewma_span=3,
                                                 vol_filter_quantile=0.8, vol_filter_window=30):
    df = df.copy()

    # Smooth volume
    df['Vol_smooth'] = df['volume'].ewm(span=vol_ewma_span, adjust=False).mean()
    df['Vol_avg_30'] = df['Vol_smooth'].rolling(window=30, min_periods=1).mean()
    df['Vol_sum_5'] = df['Vol_smooth'].rolling(window=5, min_periods=1).sum()

    # Adaptive window vector: just 2 window sizes, so create mask for volume spike
    vol_spike_mask = df['Vol_sum_5'] > vol_spike_factor * df['Vol_avg_30']
    # Create arrays of base_window and min_window
    momentum_window = np.where(vol_spike_mask, min_window, base_window).astype(int)

    # Log returns and volatility filter
    df['LogReturn'] = np.log(df['close']).diff()
    df['Volatility'] = df['LogReturn'].rolling(window=vol_filter_window, min_periods=1).std()
    vol_thresh = df['Volatility'].quantile(vol_filter_quantile)

    n = len(df)
    signals = np.zeros(n, dtype=int)

    close = df['close'].values
    vol_smooth = df['Vol_smooth'].values
    volatility = df['Volatility'].values

    for i in range(n):
        window = momentum_window[i]

        if i < window:
            continue

        if volatility[i] > vol_thresh:
            continue

        price_now = close[i]
        price_past = close[i - window]
        vol_now = vol_smooth[i]
        vol_past = vol_smooth[i - window]

        if price_past <= 0 or vol_past <= 0:
            continue

        momentum = np.log(price_now / price_past)
        vol_momentum = (vol_now / vol_past) - 1

        if momentum > momentum_thresh and vol_momentum > vol_increase_thresh:
            signals[i] = 1
        elif momentum < -momentum_thresh and vol_momentum > vol_increase_thresh:
            signals[i] = 2

    df['Signal'] = signals

    # Drop intermediate columns
    df.drop(columns=['Vol_smooth', 'Vol_avg_30', 'Vol_sum_5', 'LogReturn', 'Volatility'], inplace=True)

    return df

# === Trade Processor ===
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp = row['Date']
        close = row['close']
        open_price = row['open']
        high = row['high']
        low = row['low']
        signal = row['Filtered_Signal']

        time_str = pd.to_datetime(timestamp).strftime('%H:%M')

        # Exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif time_str >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Entry logic
        if ticker not in open_positions and start_trading_time < time_str < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# === Backtest Executor ===
def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    end_time = "15:25"
    stop_trading_time = "15:15"
    start_trading_time = "09:15"

    unique_tickers = data['Ticker'].unique()

    data_dict = {
        ticker: df.to_dict('records')
        for ticker, df in data.groupby('Ticker')
    }

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_dict[ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time
        )
        for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# === Main Execution ===

# Ensure 'Date' is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Compute signals
data = data.groupby('Ticker', group_keys=False).apply(adaptive_intraday_momentum_signal)

# Shift signals to next bar
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(int)

# Run backtest
tradebook = backtest_trades(data, 0.02, 0.02)

# Save tradebook
tradebook.to_csv("/content/drive/MyDrive/tradebook_filtered.csv", index=False)

# Show result
tradebook


In [ ]:
#data.to_csv("/content/drive/MyDrive/data_train.csv")
data

In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd

# === Metrics Calculation Function ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0

    equity = tradebook['PnL'].cumsum()
    total_profit = tradebook['PnL'].sum()

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    return ratio, total_profit

# === Hyperparameter Grid (all params included) ===
param_grid = {
    'base_window': [10, 15, 21, 34],
    'min_window': [5, 7, 9],
    'vol_spike_factor': [1.5],
    'vol_increase_thresh': [0.2],
    'momentum_thresh': [0.001, 0.002, 0.003],
    'vol_ewma_span': [2, 3],
    'vol_filter_quantile': [0.7, 0.8, 0.9],
    'vol_filter_window': [21, 30, 54],
    'tp_pct': [0.05],
    'sl_pct': [0.02]
}

# Create all combinations
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))

results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return adaptive_intraday_momentum_signal(
            group,
            base_window=param['base_window'],
            min_window=param['min_window'],
            vol_spike_factor=param['vol_spike_factor'],
            vol_increase_thresh=param['vol_increase_thresh'],
            momentum_thresh=param['momentum_thresh'],
            vol_ewma_span=param['vol_ewma_span'],
            vol_filter_quantile=param['vol_filter_quantile'],
            vol_filter_window=param['vol_filter_window']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Backtest: make sure you have this function implemented correctly
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    ratio, total_profit = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit
    }
    results.append(result)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Profit/Drawdown', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())


In [ ]:
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results.csv", index=False)